# 07 — TabPFN v2: escada de evolução E0 → E4
**TabPFN v2** (Prior Labs, Nature 2025) é um Transformer pré-treinado em milhões de bases tabulares sintéticas. Ele **não treina por gradiente**: recebe o treino como contexto e prevê em uma passada. A lógica está em `src/tabpfn_model.py`; este notebook só **chama** o módulo e registra em `reports/tables/resultados_tabpfn.csv`.

## Rodar no Colab com GPU
`Ambiente de execução > Alterar o tipo de ambiente de execução > T4 GPU`. Em CPU funciona, mas fica bem mais lento (minutos por etapa em vez de segundos). Os pesos do **v2** baixam do HuggingFace na primeira execução, **sem token e sem licença** (só as versões 2.5+ exigem).

## Regras (as mesmas do MLP e do XGBoost)
- A versão do modelo é fixada em **v2**, como no enunciado.
- **Escolhas usam somente o KS de validação** (desempate: AUROC). A validação **não entra** no modelo; serve só para escolher a etapa. O **teste só é registrado**.
- Escolhas feitas no **Fold 1**; a configuração final é reaplicada nos Folds 1, 2 e 3 com 3 sementes (a semente muda as permutações internas do ensemble).
- O KS de **treino** é otimista por construção: o modelo vê as próprias linhas de treino no contexto.
- Sem busca de hiperparâmetros: o TabPFN é feito para funcionar sem ajuste. A escada varia a **entrada**.

## Escada de evolução
| Etapa | O que muda | Variante |
|---|---|---|
| **E0_baseline** | TabPFN v2 padrão (8 membros), **todas as variáveis**, contexto = treino oficial (com oversampling) | `neural` |
| **E1_categoricas** | Categóricas tratadas pelo próprio TabPFN, sem one-hot | `arvore` |
| **E2_sem_repeticao** | Melhor variante até aqui, contexto **só com as linhas originais** (sem as cópias do oversampling) | a escolhida |
| **E3_ensemble32** | Melhor configuração até aqui, com **32 membros** no ensemble | a escolhida |
| **E4_final_3folds** | Melhor entre E0–E3 (por KS de validação), em 3 folds × 3 sementes | a escolhida |

In [ ]:
# Colab sem Git: envie o .zip do repositório no painel Arquivos (à esquerda) e rode esta célula.
# Ela descompacta em /content e entra na pasta do projeto. No VS Code (ou se o repo já estiver aberto) não faz nada.
import os, sys, zipfile
from pathlib import Path
tem_repo = any((p / "src" / "config.py").exists() for p in [Path.cwd(), *Path.cwd().parents])
if "google.colab" in sys.modules and not tem_repo:
    candidatos = [z for z in Path("/content").glob("*.zip")
                  if any(n.endswith("src/config.py") for n in zipfile.ZipFile(z).namelist())]
    if not candidatos:
        raise FileNotFoundError("Envie o .zip do repositório para /content (painel Arquivos) e rode de novo.")
    with zipfile.ZipFile(candidatos[0]) as z:
        raiz = next(n for n in z.namelist() if n.endswith("src/config.py")).rsplit("src/config.py", 1)[0]
        z.extractall("/content")
    os.chdir(Path("/content") / raiz)
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Pasta atual:", Path.cwd())

In [ ]:
# No Colab: instalar o TabPFN se faltar
import importlib.util, subprocess, sys
if importlib.util.find_spec("tabpfn") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tabpfn"], check=True)

In [ ]:
# Localiza a raiz do repositório (VS Code e Colab) e a coloca no sys.path
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Raiz do projeto:", ROOT)

In [ ]:
import warnings
from importlib.metadata import version
import pandas as pd
import matplotlib.pyplot as plt

from src import config as cfg
from src import tabpfn_model as tp
from src.metrics import (load_results, summarize_folds, evolution_table, plot_evolution,
                         plot_ks, plot_roc, plot_confusion, save_figure, save_results)

warnings.filterwarnings("ignore", category=UserWarning)
cfg.ensure_dirs()
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

# ---- Parâmetros do experimento (alterar apenas aqui) ----
FOLD_BUSCA = 1
SEEDS = (42, 43, 44)
DEVICE = tp.device_padrao()      # "cuda" no Colab com GPU

print("tabpfn:", version("tabpfn"), "| modelo:", tp.VERSAO, "| dispositivo:", DEVICE)
if DEVICE == "cpu":
    print("Aviso: sem GPU. Funciona, mas vai demorar. No Colab, ative a T4.")

In [ ]:
def registrar(config, variante, etapa, obs):
    res, saidas = tp.avaliar_configuracao(config, variante, FOLD_BUSCA, SEEDS, etapa=etapa, obs=obs, device=DEVICE)
    save_results(res, cfg.REPORTS_TABLES)
    resumo = res.groupby("conjunto")[["ks", "auroc", "mse", "f1"]].mean().reindex(["treino", "validacao", "teste"])
    print(f"{etapa} | {variante} | {tp.config_to_dict(config)}")
    display(resumo.round(4))
    return res, saidas

## E0 — Baseline: TabPFN v2 padrão, todas as variáveis
Mesma entrada do MLP (`neural`: one-hot + Min-Max) e mesmo treino (com oversampling).

In [ ]:
X_tr, y_tr = tp.load_xy(FOLD_BUSCA, "neural")[0]["treino"]
print(f"Contexto: {X_tr.shape[0]} linhas x {X_tr.shape[1]} colunas (limite do v2: 10.000 x 500)")
res_e0, saidas_e0 = registrar(tp.BASELINE, "neural", "E0_baseline", "TabPFN v2 padrão, todas as variáveis")

## E1 — Categóricas nativas (`arvore`)
Em vez de one-hot, as 16 categóricas entram como códigos e o TabPFN é avisado de quais colunas são categóricas (categorias aprendidas só no treino).

In [ ]:
res_e1, _ = registrar(tp.BASELINE, "arvore", "E1_categoricas", "categóricas tratadas pelo TabPFN, sem one-hot")

## E2 — Contexto sem as cópias do oversampling
O oversampling repete linhas de churn para chegar a 50/50. Para o TabPFN, cópias idênticas no contexto não trazem informação nova e podem distorcer o que ele aprendeu no pré-treino. Aqui o contexto usa **só as linhas originais** do treino (~2.350). A variante é a melhor entre E0 e E1 pela validação. As métricas de treino continuam no treino oficial, para comparar com os outros modelos.

In [ ]:
resultados = load_results(cfg.REPORTS_TABLES)
etapa_ref, variante_ref, comp = tp.melhor_etapa(resultados, ["E0_baseline", "E1_categoricas"], FOLD_BUSCA)
display(comp.round(4))
print(f"Variante escolhida pela validação: {variante_ref} ({etapa_ref})")
cfg_e2 = tp.TabPFNConfig(n_estimators=8, sem_repeticao=True)
res_e2, _ = registrar(cfg_e2, variante_ref, "E2_sem_repeticao", "contexto só com linhas originais do treino")

## E3 — Ensemble maior (32 membros)
Mais membros = mais permutações de colunas e pré-processamentos médias juntas. Tende a estabilizar a previsão, ao custo de tempo.

In [ ]:
resultados = load_results(cfg.REPORTS_TABLES)
etapa_ref, variante_ref, comp = tp.melhor_etapa(resultados, ["E0_baseline", "E1_categoricas", "E2_sem_repeticao"], FOLD_BUSCA)
display(comp.round(4))
cfg_ref = tp.config_da_etapa(resultados, etapa_ref, FOLD_BUSCA)
cfg_e3 = tp.TabPFNConfig(n_estimators=32, sem_repeticao=cfg_ref.sem_repeticao)
print(f"Base: {etapa_ref} ({variante_ref}) -> 32 membros")
res_e3, _ = registrar(cfg_e3, variante_ref, "E3_ensemble32", f"{etapa_ref} com 32 membros")

## E4 — Resultado final nos 3 folds
Escolhemos entre E0–E3 **pelo KS médio de validação no Fold 1** e reaplicamos nos Folds 1, 2 e 3 com 3 sementes.

In [ ]:
resultados = load_results(cfg.REPORTS_TABLES)
etapas = ["E0_baseline", "E1_categoricas", "E2_sem_repeticao", "E3_ensemble32"]
etapa_vencedora, variante_final, comparacao = tp.melhor_etapa(resultados, etapas, FOLD_BUSCA)
print("Comparação (KS e AUROC de VALIDAÇÃO, Fold 1, média das sementes):")
display(comparacao.round(4))
config_final = tp.config_da_etapa(resultados, etapa_vencedora, FOLD_BUSCA)
tp.save_config(config_final, cfg.REPORTS_TABLES / "tabpfn_melhor.json", variante_final)
print(f"Vencedora: {etapa_vencedora} | {variante_final} | {tp.config_to_dict(config_final)}")

res_e4, saidas_e4 = tp.etapa_final(config_final, variante_final, etapa="E4_final_3folds", folds=(1, 2, 3),
                                   seeds=SEEDS, device=DEVICE, out_dir=cfg.REPORTS_TABLES)

In [ ]:
resultados = load_results(cfg.REPORTS_TABLES)
res_tp = resultados[resultados["modelo"] == "tabpfn"]
for conj in ("validacao", "teste"):
    print(f"\n=== E4 — {conj} (média ± desvio entre folds) ===")
    display(summarize_folds(res_tp[res_tp["etapa"] == "E4_final_3folds"], conj).round(4))

por_fold = (res_tp[(res_tp["etapa"] == "E4_final_3folds") & (res_tp["conjunto"] == "teste")]
            .groupby("fold")[["ks", "auroc", "f1", "recall", "precisao", "mse", "log_loss"]].mean())
print("\nTeste por fold (média das sementes):")
display(por_fold.round(4))

In [ ]:
# Gráficos do teste (Fold 1, 1ª semente)
y_te, p_te = saidas_e4[1][SEEDS[0]]["scores"]["teste"]
fig, axs = plt.subplots(1, 3, figsize=(15, 4.2), layout="constrained")
plot_ks(y_te, p_te, "TabPFN v2 final — teste, Fold 1", ax=axs[0])
plot_roc(y_te, p_te, "ROC — teste, Fold 1", ax=axs[1])
plot_confusion(y_te, p_te, title="Matriz de confusão — teste, Fold 1", ax=axs[2])
save_figure(fig, "tabpfn_final_teste_fold1", cfg.REPORTS_FIGURES)
plt.show()

## Evolução do desempenho
KS por etapa no **Fold 1**. A escolha usou só a validação; o teste aparece como registro.

In [ ]:
resultados = load_results(cfg.REPORTS_TABLES)
fig = plot_evolution(resultados, "tabpfn", metric="ks", fold=FOLD_BUSCA)
save_figure(fig, "tabpfn_evolucao_ks_fold1", cfg.REPORTS_FIGURES)
plt.show()
tabela_evolucao = evolution_table(resultados, "tabpfn", fold=FOLD_BUSCA, metrics=("ks", "auroc", "mse", "log_loss", "f1"))
tabela_evolucao.to_csv(cfg.REPORTS_TABLES / "tabpfn_evolucao_fold1.csv", index=False)
tabela_evolucao.round(4)

## Comparação dos modelos (resultado final, teste, 3 folds)

In [ ]:
finais = {"mlp": "E4_final_3folds", "xgboost": "E3_final_3folds", "tabpfn": "E4_final_3folds"}
mask = False
for m, e in finais.items():
    mask = mask | ((resultados["modelo"] == m) & (resultados["etapa"] == e))
summarize_folds(resultados[mask], "teste")[["modelo", "ks_media", "ks_dp", "auroc_media", "auroc_dp", "f1_media",
                                            "recall_media", "precisao_media", "folds", "sementes"]].round(4)

In [ ]:
# Colab: junta os resultados do TabPFN num .zip e baixa (para subir no GitHub pelo navegador)
import sys, zipfile
from pathlib import Path
if "google.colab" in sys.modules:
    arquivos = sorted(cfg.REPORTS_TABLES.glob("*tabpfn*")) + sorted(cfg.REPORTS_FIGURES.glob("tabpfn_*"))
    destino = Path("/content/resultados_tabpfn.zip")
    with zipfile.ZipFile(destino, "w") as z:
        for a in arquivos:
            z.write(a, a.relative_to(cfg.ROOT))
    print("Arquivos no zip:", [str(a.relative_to(cfg.ROOT)) for a in arquivos])
    from google.colab import files
    files.download(str(destino))

## Registro do processo
_Preencher depois de rodar (mesmo formato do 05_mlp)._